# 05 — Synthetic benchmark v0.1: pilot generation walkthrough

Runs the reusable generator package (`src/boamp/synthetic/`) end-to-end for
one scenario and inspects the result. All generation logic lives in
`src/boamp/synthetic/*.py`; this notebook only calls it and displays outputs
(spec: "Do not place core generation logic only in the notebook").

Adapts the gold-standard-and-corruption framework of Lam et al. (2024) to
public-procurement recurrence linkage: a clean latent world (buyers,
establishments, needs, cycles, true relations) is generated first, then
BOAMP-like publication notices are corrupted into `observed_notices`, while
the truth tables are retained separately and never exposed to Layer 1/Layer 2.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd

from boamp.synthetic.parameters import load_calibration_parameters
from boamp.synthetic.scenarios import VALID_SCENARIOS, load_benchmark_defaults, load_scenario
from boamp.synthetic.pipeline import generate_clean_world, generate_observed_world, write_pilot_outputs

pd.set_option("display.max_colwidth", 120)
print("Available scenarios:", VALID_SCENARIOS)

## 1. Load configuration and select a scenario

In [ ]:
SCENARIO_ID = "central_provisional"

calib = load_calibration_parameters(PROJECT_ROOT)
defaults = load_benchmark_defaults(PROJECT_ROOT)
scenario = load_scenario(PROJECT_ROOT, SCENARIO_ID)

print("benchmark_id:", defaults.benchmark_id)
print("target_n_buyers:", defaults.target_n_buyers, " target_n_notices (approx):", defaults.target_n_notices)
print("seeds:", defaults.seed.latent_world_seed, defaults.seed.corruption_seed)
print()
print(f"Scenario: {scenario.display_name!r} ({scenario.scenario_id})")
print("Purpose:", scenario.purpose.strip())
print("base_recurrence_propensity:", scenario.recurrence.base_recurrence_propensity)
print("identifiers.siret_missing_rate:", scenario.identifiers.siret_missing_rate)
print("cpv.missing_rate:", scenario.cpv.missing_rate)

## 2. Run a small pilot (n_buyers below the full 2,000-buyer default, for a fast notebook run)

In [ ]:
world = generate_clean_world(SCENARIO_ID, PROJECT_ROOT, n_buyers=300, world_seed=defaults.seed.latent_world_seed)
print("Clean-world structural validation:", "PASS" if world["_validation"].passed else world["_validation"].failures())

observed, corruption_log = generate_observed_world(world, SCENARIO_ID, PROJECT_ROOT, corruption_seed=defaults.seed.corruption_seed)
print("observed_notices rows:", len(observed), " corruption_log rows:", len(corruption_log))

## 3. Row counts

In [ ]:
row_counts = {k: len(v) for k, v in world.items() if k != "_validation"}
row_counts["observed_notices"] = len(observed)
row_counts["corruption_log"] = len(corruption_log)
pd.Series(row_counts, name="n_rows").to_frame()

## 4. Structural checks

In [ ]:
from boamp.synthetic.validation import run_full_structural_validation

result = run_full_structural_validation(world)
pd.Series(result.checks, name="passed").to_frame()

## 5. Clean -> corrupted examples

In [ ]:
clean_idx = world["clean_notices"].set_index("notice_id_synthetic")
sample_ids = observed["notice_id_synthetic"].sample(5, random_state=1).tolist()

compare_cols = ["buyer_siret_raw", "buyer_siren_raw", "buyer_name_raw", "cpv_clean",
                "declared_duration_months", "objet_clean", "linked_call_notice_id"]
clean_compare_cols = ["siret_true", "siren_true", "buyer_name_true", "cpv_true",
                      "duration_true_months", "objet_true", "linked_call_notice_id_true"]

rows = []
for nid in sample_ids:
    c = clean_idx.loc[nid]
    o = observed[observed["notice_id_synthetic"] == nid].iloc[0]
    for cc, oc in zip(clean_compare_cols, compare_cols):
        rows.append({"notice_id": nid, "field": oc, "clean": c[cc], "observed": o[oc]})
pd.DataFrame(rows)

## 6. Corruption type summary

In [ ]:
corruption_log["corruption_type"].value_counts().to_frame("n_events")

In [ ]:
corruption_log.groupby("field")["corruption_type"].value_counts().unstack(fill_value=0)

## 7. Recurrence (NEXT_CYCLE) and NO_SUCCESSOR examples

In [ ]:
next_cycle_examples = world["true_relations"][world["true_relations"]["relation_type"] == "NEXT_CYCLE"].head(5)
next_cycle_examples[["source_cycle_id", "target_cycle_id", "true_gap_months", "source_expected_end", "target_start"]]

In [ ]:
no_successor_examples = world["true_relations"][world["true_relations"]["relation_type"] == "NO_SUCCESSOR"].head(5)
no_successor_examples[["source_cycle_id", "target_cycle_id", "source_expected_end"]]

In [ ]:
print("Relation type mix:")
print(world["true_relations"]["relation_type"].value_counts(normalize=True))
gaps = world["true_relations"].loc[world["true_relations"]["relation_type"] == "NEXT_CYCLE", "true_gap_months"]
print(f"\ntrue_gap_months beyond the real pipeline's 6-month window: {(gaps > 6).mean():.1%} of NEXT_CYCLE edges")

## 8. Full pilot outputs

The full 2,000-buyer pilot for all three scenarios (`clean_sanity`,
`central_provisional`, `adverse_identity`) was already generated via
`boamp.synthetic.pipeline.generate_pilot` and is on disk at:

`data/processed/synthetic_benchmark/v0_1_provisional/<scenario>/world_001/corruption_001/`

Each directory contains `latent_buyers.parquet`, `latent_establishments.parquet`,
`latent_needs.parquet`, `latent_cycles.parquet`, `true_relations.parquet`,
`notice_family_membership.parquet`, `clean_notices.parquet`,
`observed_notices.parquet`, `corruption_log.parquet`, and
`generation_metadata.json`. Structural validation results for all three are
in `reports/generated/synthetic_benchmark/v0_1_clean_world_validation.md`.
Fidelity validation against the real corpus is in
`06_synthetic_fidelity_validation.ipynb`.


In [ ]:
import json
for scen in ["clean_sanity", "central_provisional", "adverse_identity"]:
    meta_path = PROJECT_ROOT / "data/processed/synthetic_benchmark/v0_1_provisional" / scen / "world_001/corruption_001/generation_metadata.json"
    meta = json.loads(meta_path.read_text())
    print(scen, "-> validation:", meta["validation_status"], " row_counts:", meta["row_counts"])